# Agentic Multimodal Fake Review Detection & Generative Trust Scoring

This notebook implements an agentic workflow that leverages Google GenAI models to process multimodal inputs (text, images, metadata) of online reviews. It determines the probability of fake reviews and generates a granular, explainable **Generative Trust Score**.

In [2]:
# Install necessary libraries
!pip install google-genai pillow pydantic

### 1. Initialize Gemini Client
Configure your Google GenAI Client using secrets manager keys.

In [4]:
from google import genai
from google.colab import userdata
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    client = genai.Client(api_key=GOOGLE_API_KEY)
    print("Gemini client successfully initialized!")
except Exception as e:
    print(f"Could not initialize client automatically. Ensure GOOGLE_API_KEY is in your secrets: {e}")

Gemini client successfully initialized!


### 2. Define Multimodal Schema & Data Structures
We define structured output using Pydantic to ensure our agent's analysis is strictly formatted.

In [5]:
from pydantic import BaseModel, Field
from typing import List, Optional

class FeatureAnalysis(BaseModel):
    text_sentiment_consistency: str = Field(description="Analysis of whether the text sentiment matches the visual signals.")
    image_genuineness_check: str = Field(description="Evaluation of whether the review images look stock, duplicated, generated, or contextually relevant.")
    metadata_anomalies: List[str] = Field(description="List of potential red flags in metadata (e.g., mismatching date, high posting frequency).")

class TrustEvaluation(BaseModel):
    fake_review_probability: float = Field(description="Estimated probability that this review is fake (0.0 to 1.0).")
    generative_trust_score: float = Field(description="Calculated overall trust score for the user profile (0.0 to 100.0).")
    key_reasons: List[str] = Field(description="Top reasons driving this scoring decision.")
    actionable_recommendation: str = Field(description="Recommended action (e.g., Flag, Approve, Pending Human Verification).")
    analysis_details: FeatureAnalysis

### 3. Agentic Scorer Engine
Now, we build the core function utilizing `gemini-2.5-flash` to evaluate multimodal inputs against our structured schema.

In [19]:
from PIL import Image
import io

def run_fake_review_agent(review_text: str, review_image: Optional[Image.Image] = None, metadata: Optional[dict] = None) -> TrustEvaluation:
    """
    Evaluates text, image, and metadata context to assess authenticity and calculate a trust score.
    """
    prompt = f"""
    You are an AI Trust and Safety Agent specialized in e-commerce and local business platforms.
    Analyze the following review details, along with any attached media, to identify signs of automated creation, fake positive/negative brigade campaigns, or mismatched visual-text signals.

    Review Text: {review_text}
    Metadata: {metadata if metadata else 'None'}
    """

    contents = [prompt]
    if review_image:
        contents.append(review_image)

    response = client.models.generate_content(
        model='gemini-3.5-flash-lite',
        contents=contents,
        config={
            'response_mime_type': 'application/json',
            'response_schema': TrustEvaluation,
            'temperature': 0.2
        }
    )
    return TrustEvaluation.model_validate_json(response.text)

### 4. Run a Demo Analysis
Let's run a test analysis utilizing a mock sample review.

In [20]:
# Creating a mock image (blank gray block)
img = Image.new('RGB', (200, 200), color = 'gray')

mock_text = "This restaurant is the absolute worst. Horrible experience, but the picture is of a beautiful empty dining table with pristine decorations from an advertisement page."
mock_metadata = {
    "user_total_reviews": 1,
    "account_age_days": 0,
    "review_rating": 1.0
}

try:
    evaluation = run_fake_review_agent(review_text=mock_text, review_image=img, metadata=mock_metadata)
    import json
    print(json.dumps(evaluation.model_dump(), indent=2))
except Exception as e:
    print(f"Review evaluation failed: {e}\nMake sure your API key is correctly configured.")

{
  "fake_review_probability": 0.95,
  "generative_trust_score": 10.0,
  "key_reasons": [
    "Brand new account created 0 days ago with zero prior history.",
    "Text explicitly points out a mismatch by referencing an advertisement-style picture.",
    "High likelihood of automated creation or a malicious review campaign."
  ],
  "actionable_recommendation": "Flag",
  "analysis_details": {
    "text_sentiment_consistency": "Highly inconsistent, as the text describes a negative experience while referencing pristine promotional imagery.",
    "image_genuineness_check": "The image appears to be a stock photo or promotional advertisement asset rather than a genuine user-generated photo of a dining experience.",
    "metadata_anomalies": [
      "Account age is 0 days",
      "User total reviews is exactly 1",
      "First-time reviewer posting an extreme negative rating with questionable media"
    ]
  }
}


### 5. Running with an Established Profile (Alternative Metadata)
Now we will evaluate a review coming from an established user profile to observe the difference in the trust score.

In [21]:
from PIL import Image

# Simulate a real-looking product image (using a warm colored block representing a product)
established_user_img = Image.new('RGB', (200, 200), color = 'peru')

established_user_text = "I bought this wooden table last week. The finish is fantastic and it fits perfectly in my living room. Very satisfied with the build quality!"

established_metadata = {
    "user_total_reviews": 42,
    "account_age_days": 730,  # 2 years old account
    "verified_purchase": True,
    "review_rating": 5.0
}

try:
    evaluation_established = run_fake_review_agent(
        review_text=established_user_text,
        review_image=established_user_img,
        metadata=established_metadata
    )
    import json
    print(json.dumps(evaluation_established.model_dump(), indent=2))
except Exception as e:
    print(f"Review evaluation failed: {e}\nMake sure your API key is correctly configured.")

{
  "fake_review_probability": 0.12,
  "generative_trust_score": 88.5,
  "key_reasons": [
    "Account is two years old with a healthy history of 42 reviews.",
    "Verified purchase status confirms a real transaction.",
    "Text sentiment is natural and directly aligns with the product purchased."
  ],
  "actionable_recommendation": "Approve",
  "analysis_details": {
    "text_sentiment_consistency": "High consistency between the positive review text and the 5.0 rating.",
    "image_genuineness_check": "Attached image is a solid color swatch likely representing wood tone, contextually matching the wooden table.",
    "metadata_anomalies": []
  }
}
